# 🩺 MediSpark — Colab LLM Server

Runs **Ollama + a 7B model** on Colab's free T4 GPU and exposes it via an **ngrok tunnel**  
so your local MediSpark Flask app can use it instead of `gemma3:1b`.

### Setup steps
1. **Runtime → Change runtime type → T4 GPU** (free tier)
2. Get a free ngrok auth token at https://dashboard.ngrok.com/signup  
   paste it in Cell 4
3. Run all cells top-to-bottom
4. Copy the `OLLAMA_URL` printed by Cell 6 into your local `.env`
5. Set `OLLAMA_MODEL=llama3.2:3b` (or whichever model you pulled) in `.env`
6. Restart Flask — done

### Model options (pick one in Cell 5)
| Model | VRAM | Quality | Speed via tunnel |
|-------|------|---------|------------------|
| `llama3.2:3b` | ~3 GB | ★★★★☆ | Fast |
| `mistral:7b-instruct-v0.3` | ~5 GB | ★★★★★ | Medium |
| `phi3:mini` | ~2 GB | ★★★☆☆ | Fastest |

> **Keep this tab open.** If Colab disconnects, the tunnel dies and Flask falls back to rule-based mode automatically.

In [11]:
!sudo apt-get update
!sudo apt-get install zstd

Hit:1 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ InRelease
Hit:2 https://developer.download.nvidia.com/compute/cuda/repos/ubuntu2204/x86_64  InRelease
Hit:3 https://cli.github.com/packages stable InRelease                         
Hit:4 http://archive.ubuntu.com/ubuntu jammy InRelease                         
Hit:5 http://security.ubuntu.com/ubuntu jammy-security InRelease               
Hit:6 https://r2u.stat.illinois.edu/ubuntu jammy InRelease                     
Hit:7 http://archive.ubuntu.com/ubuntu jammy-updates InRelease                 
Hit:8 http://archive.ubuntu.com/ubuntu jammy-backports InRelease               
Hit:9 https://ppa.launchpadcontent.net/deadsnakes/ppa/ubuntu jammy InRelease
Hit:10 https://ppa.launchpadcontent.net/graphics-drivers/ppa/ubuntu jammy InRelease
Hit:11 https://ppa.launchpadcontent.net/ubuntugis/ppa/ubuntu jammy InRelease
Reading package lists... Done
W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r

In [12]:
# ── Cell 1: Install Dependencies and Ollama ────────────────────────────────────
!sudo apt-get update -y
!sudo apt-get install zstd -y
!curl -fsSL https://ollama.com/install.sh | sh

Hit:1 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ InRelease
Hit:2 https://cli.github.com/packages stable InRelease                         
Hit:3 https://developer.download.nvidia.com/compute/cuda/repos/ubuntu2204/x86_64  InRelease
Hit:4 https://r2u.stat.illinois.edu/ubuntu jammy InRelease                     
Hit:5 http://security.ubuntu.com/ubuntu jammy-security InRelease               
Hit:6 http://archive.ubuntu.com/ubuntu jammy InRelease              
Hit:7 http://archive.ubuntu.com/ubuntu jammy-updates InRelease
Hit:8 https://ppa.launchpadcontent.net/deadsnakes/ppa/ubuntu jammy InRelease
Hit:9 http://archive.ubuntu.com/ubuntu jammy-backports InRelease
Hit:10 https://ppa.launchpadcontent.net/graphics-drivers/ppa/ubuntu jammy InRelease
Hit:11 https://ppa.launchpadcontent.net/ubuntugis/ppa/ubuntu jammy InRelease
Reading package lists... Done
W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu jammy InRelease

In [13]:
# ── Cell 2: Start Ollama server (background) ─────────────────────────────────
import subprocess, time, os

env = os.environ.copy()
env["OLLAMA_HOST"]    = "0.0.0.0"   # accept connections from ngrok
env["OLLAMA_ORIGINS"] = "*"          # allow cross-origin requests

proc = subprocess.Popen(
    ["ollama", "serve"],
    env=env,
    stdout=subprocess.DEVNULL,
    stderr=subprocess.DEVNULL,
)
time.sleep(4)  # wait for server to start
print("✅ Ollama server started (PID", proc.pid, ")")

✅ Ollama server started (PID 8994 )


In [ ]:
# ── Cell 3: Pull the model ────────────────────────────────────────────────────
# Change this to 'mistral:7b-instruct-v0.3' for higher quality (takes ~3 min)
MODEL = "qwen2.5:7b"

!ollama pull {MODEL}
print(f"\n✅ Model '{MODEL}' ready")



✅ Model 'llama3.2:3b' ready


In [15]:
# ── Cell 4: Configure ngrok ───────────────────────────────────────────────────
# Get your free token at https://dashboard.ngrok.com/signup
NGROK_AUTH_TOKEN = "3DI4etVINJwC9qzBtAW2uGJ6QM4_2WfVJU4f9rgqU4SAknycR"

!pip install pyngrok -q
from pyngrok import ngrok, conf
conf.get_default().auth_token = NGROK_AUTH_TOKEN
print("✅ ngrok configured")

✅ ngrok configured


In [16]:
# ── Cell 5: Open tunnel & print config ───────────────────────────────────────
tunnel = ngrok.connect(11434, "http")
public_url = tunnel.public_url

print("═" * 60)
print("🚇 Tunnel active — add these to your local .env file:")
print("═" * 60)
print(f"OLLAMA_URL={public_url}")
print(f"OLLAMA_MODEL={MODEL}")
print("OLLAMA_TIMEOUT=60")
print("═" * 60)
print("Then restart Flask on your local machine.")

════════════════════════════════════════════════════════════
🚇 Tunnel active — add these to your local .env file:
════════════════════════════════════════════════════════════
OLLAMA_URL=https://slinky-upheaval-comfy.ngrok-free.dev
OLLAMA_MODEL=llama3.2:3b
OLLAMA_TIMEOUT=60
════════════════════════════════════════════════════════════
Then restart Flask on your local machine.


In [ ]:
import requests, time

print("Warming up model — first load takes 1-3 min, please wait...")
r = requests.post(
    "http://localhost:11434/api/chat",
    json={
        "model": MODEL,
        "messages": [{"role": "user", "content": "hi"}],
        "stream": False,
        "options": {"num_predict": 5},
    },
    timeout=300,   # 5 min — plenty for first GPU load
)
print("Warm-up done:", r.json().get("message", {}).get("content", ""))


In [17]:
# ── Cell 6: Smoke test ────────────────────────────────────────────────────────
import requests, json

resp = requests.post(
    f"{public_url}/api/chat",
    json={
        "model": MODEL,
        "messages": [{"role": "user", "content": "Reply with exactly: MediSpark ready"}],
        "stream": False,
        "options": {"num_predict": 10},
    },
    timeout=60,
)
reply = resp.json().get("message", {}).get("content", "")
print("Model reply:", reply)
print("✅ Tunnel is working!" if resp.ok else "❌ Something went wrong:", resp.status_code)

Model reply: MediSpark ready
✅ Tunnel is working! 200


In [18]:
# ── Cell 7: Keep-alive (run this to prevent Colab from disconnecting) ─────────
# Colab disconnects after ~90 min of inactivity. This loop pings the model
# every 20 minutes to keep the session alive.
import time, datetime

print("Keep-alive running. Stop this cell to shut down.")
while True:
    try:
        r = requests.post(
            "http://localhost:11434/api/chat",
            json={"model": MODEL, "messages": [{"role": "user", "content": "ping"}],
                  "stream": False, "options": {"num_predict": 1}},
            timeout=30,
        )
        print(f"[{datetime.datetime.now():%H:%M}] ping ok — model alive")
    except Exception as e:
        print(f"[{datetime.datetime.now():%H:%M}] ping failed: {e}")
    time.sleep(20 * 60)  # ping every 20 minutes

Keep-alive running. Stop this cell to shut down.
[21:58] ping ok — model alive


KeyboardInterrupt: 